In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

np.random.seed(15)

X, y_labels = make_moons(n_samples=200, noise=0.1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y_labels, test_size=0.2, random_state=42)

class Node:
    def __init__(self, dim, threshold, left, right, decision) -> None:
        self.dim = dim
        self.threshold = threshold
        self.left = left
        self.right = right
        self.decision = decision

def mode(x: np.ndarray):
    return np.argmax(np.bincount(x))

def gini(y: np.ndarray):
    _, counts = np.unique(y, return_counts=True)
    return 1 - np.sum((counts/len(y))**2)


def build_tree(X: np.ndarray, y: np.ndarray, depth=0, min_samples_split=5, max_depth=10):
    N = len(X)
    if X.shape[0] < min_samples_split or depth > max_depth or gini(y) == 0:
        return Node(None, None, None, None, mode(y))
    best_dim, best_threshold, best_gini = 0, 0.0, 1
    for dim in range(X.shape[1]):
        X_dim = np.sort(X[:, dim])
        for i in range(X.shape[0] - 1):
            threshold = (X_dim[i] + X_dim[i+1])/2
            left_mask, right_mask = X[:, dim] < threshold, X[:, dim] >= threshold
            total_gini = (np.sum(left_mask)/N) * gini(y[left_mask]) + (np.sum(right_mask)/N) * gini(y[right_mask])
            if total_gini < best_gini:
                best_gini = total_gini
                best_dim = dim
                best_threshold = threshold
    left_mask, right_mask = X[:, best_dim] < best_threshold, X[:, best_dim] >= best_threshold
    return Node(
        dim=best_dim,
        threshold=best_threshold,
        left=build_tree(X[left_mask], y[left_mask], depth=depth+1, min_samples_split=min_samples_split, max_depth=max_depth),
        right=build_tree(X[right_mask], y[right_mask], depth=depth+1, min_samples_split=min_samples_split, max_depth=max_depth),
        decision=None
    )

def predict(X: np.ndarray, tree):
    def pred(x, node):
        if node.decision is not None:
            return node.decision
        if x[node.dim] >= node.threshold:
            return pred(x, node.right)
        else:
            return pred(x, node.left)

    preds = []
    for x in X:
        preds.append(pred(x, tree))
    return np.array(preds)


tree = build_tree(X_train, y_train)
my_preds = predict(X_test, tree)

sk_tree = DecisionTreeClassifier(
    criterion='gini',
    max_depth=10,
    min_samples_split=5,
    random_state=42,
)
sk_tree.fit(X_train, y_train)
sk_preds = sk_tree.predict(X_test)

# Accuracy comparison
agreement = np.mean(my_preds == sk_preds)
print(f"agreement: {agreement:.3f}")
